# Лабораторная работа 8: дискретно-событийная модель SIR

В базовом сценарии моделируется эпидемия в полностью смешивающейся
популяции. Каждый восприимчивый агент через экспоненциально распределённые
интервалы времени выбирает случайный контакт. При встрече с инфицированным
заражение происходит с вероятностью `beta`.

In [1]:
using DrWatson
@quickactivate "project"

ENV["GKSwstype"] = "100"

using CSV
using Plots

include(srcdir("SIRDESLab08.jl"))
using .SIRDESLab08

mkpath(datadir("sims"))
mkpath(plotsdir())

"/home/emkurilkoryumin/work/study/2026-1/2026-1==study--simulation-modeling/2026-1--study--simulation-modeling/labs/lab08/project/plots"

## Параметры базового сценария

Начальное состояние `u0 = [S0, I0, R0]`. Параметры модели:
`beta` — вероятность передачи при контакте, `contacts` — частота контактов,
`gamma` — интенсивность выздоровления.

In [2]:
tmax = 40.0
u0 = [990, 10, 0]
parameters = [0.05, 10.0, 0.25]

result = run_sir(
    u0,
    parameters;
    tmax = tmax,
    seed = 1234,
    scenario = "base",
)

CSV.write(datadir("sims", "sir_990_10_0.05_10.0_0.25.csv"), result.data)
CSV.write(datadir("sims", "sir_base_summary.csv"), result.summary)

println("Базовый сценарий DES-SIR")
println(result.summary)

Базовый сценарий DES-SIR
1×14 DataFrame
 Row │ scenario  parameter  value    beta     contacts  gamma    recovery_mode  peak_infected  peak_time  final_susceptible  final_infected  final_recovered  affected_share  events 
     │ String    String     Float64  Float64  Float64   Float64  String         Int64          Float64    Int64              Int64           Int64            Float64         Int64  
─────┼───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   1 │ base      base           NaN     0.05      10.0     0.25  exponential              152    18.4475                230              11              759            0.77    1519


## График стохастической траектории

In [3]:
plot(
    result.data.t,
    result.data.S;
    label = "S",
    linewidth = 2,
    xlabel = "Время",
    ylabel = "Численность",
    title = "Дискретно-событийная SIR-модель",
)
plot!(result.data.t, result.data.I; label = "I", linewidth = 2)
plot!(result.data.t, result.data.R; label = "R", linewidth = 2)
savefig(plotsdir("sir_des.png"))

"/home/emkurilkoryumin/work/study/2026-1/2026-1==study--simulation-modeling/2026-1--study--simulation-modeling/labs/lab08/project/plots/sir_des.png"

## Сравнение с детерминированной моделью

Для контроля стохастическая траектория сопоставляется с численным решением
системы ОДУ SIR. ОДУ интегрируются методом Рунге-Кутты четвёртого порядка.

In [4]:
ode = solve_deterministic_sir(u0, parameters; tmax = tmax)

plot(
    result.data.t,
    result.data.I;
    label = "DES: I(t)",
    linewidth = 2,
    xlabel = "Время",
    ylabel = "Инфицированные",
    title = "Стохастическая DES и детерминированная SIR-модель",
)
plot!(ode.t, ode.I; label = "ОДУ: I(t)", linewidth = 2, linestyle = :dash)
savefig(plotsdir("sir_des_vs_ode.png"))

CSV.write(datadir("sims", "sir_deterministic_ode.csv"), ode)

"/home/emkurilkoryumin/work/study/2026-1/2026-1==study--simulation-modeling/2026-1--study--simulation-modeling/labs/lab08/project/data/sims/sir_deterministic_ode.csv"